# RVC → Piper Training Studio — Google Colab

This notebook runs the headless RVC → Piper pipeline on a Google Colab NVIDIA GPU.

It supports:
- Google Drive persistence for datasets/checkpoints/final models;
- RVC `.pth` plus optional `.index`;
- pitch `+12` or any other pitch;
- high-pitch audio cleanup before Piper training;
- warm-started Piper training and resume from `last.ckpt`;
- **selectable base Piper voices** instead of forcing Alba;
- final `.onnx` + `.onnx.json` export and an inline TTS test.

Choose **Runtime → Change runtime type → GPU** before starting.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No Colab GPU is active. Change the runtime type to GPU and reconnect.")
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))


## 1. Clone/update the Studio repo

This always pulls the latest `main` branch.


In [ ]:
import os, subprocess

REPO = "/content/RVC-to-Piper-Training-APP"
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone",
        "https://github.com/NekoSuneVR/RVC-to-Piper-Training-APP.git",
        REPO,
    ], check=True)

os.chdir(REPO)
print("Repo:", REPO)


## 2. Install the Colab runtime

This creates isolated Python 3.12 RVC/Piper environments under `/content/rvc-piper-runtime`,
builds Piper's native eSpeak/alignment extensions, downloads RMVPE/HuBERT, and installs the default
**Alba Medium** Piper base voice.

You can choose a different base Piper voice below. Extra Piper voices are downloaded only when selected.
Training data, checkpoints and final models remain in Google Drive.


In [ ]:
import subprocess

result = subprocess.run(
    ["bash", "colab/setup_colab.sh", "/content/RVC-to-Piper-Training-APP"],
    text=True,
)
if result.returncode:
    print("\nSetup failed. Last 120 log lines:\n")
    subprocess.run(["tail", "-n", "120", "/content/rvc-piper-runtime/setup.log"])
    raise RuntimeError(f"Colab setup failed with exit code {result.returncode}")


## 3. Build settings

Put your RVC files in Drive, for example:

```text
MyDrive/RVC-Piper-Colab/models/my-voice.pth
MyDrive/RVC-Piper-Colab/models/my-voice.index
```

### Base Piper voice

`Alba Medium` is the default. You can select `Amy Medium`, enter **any Piper voice key** from the
Piper voices collection (for example `en_US-ryan-medium`), or provide your own `.onnx` +
`.onnx.json` from Google Drive.

The selected base voice is only used to generate the text/audio dataset before RVC `+12` conversion.


In [ ]:
#@title Build settings

voice_name = "en_GB-rvc-custom-medium" #@param {type:"string"}
rvc_model = "/content/drive/MyDrive/RVC-Piper-Colab/models/my-voice.pth" #@param {type:"string"}
rvc_index = "" #@param {type:"string"}

base_piper_voice = "Alba Medium (UK) - en_GB-alba-medium" #@param ["Alba Medium (UK) - en_GB-alba-medium", "Amy Medium (US) - en_US-amy-medium", "Other Piper voice key", "Custom ONNX + JSON from Drive"]
other_piper_voice_key = "en_US-ryan-medium" #@param {type:"string"}
custom_base_model = "/content/drive/MyDrive/RVC-Piper-Colab/base-voices/custom.onnx" #@param {type:"string"}
custom_base_config = "/content/drive/MyDrive/RVC-Piper-Colab/base-voices/custom.onnx.json" #@param {type:"string"}

pitch = 12 #@param {type:"integer"}
index_rate = 0.75 #@param {type:"number"}
protect = 0.33 #@param {type:"number"}
f0_method = "rmvpe" #@param ["rmvpe", "pm"]

prompt_file = "/content/RVC-to-Piper-Training-APP/data/piper_training_prompts.txt" #@param {type:"string"}
prompt_limit = 120 #@param {type:"integer"}

batch_size = 8 #@param {type:"integer"}
max_epochs = 1000 #@param {type:"integer"}
checkpoint_every = 5 #@param {type:"integer"}
num_workers = 2 #@param {type:"integer"}

drive_root = "/content/drive/MyDrive/RVC-Piper-Colab" #@param {type:"string"}
generate_dataset = True #@param {type:"boolean"}
resume_if_possible = True #@param {type:"boolean"}

print("Voice:", voice_name)
print("RVC:", rvc_model)
print("Pitch:", pitch)
print("Base Piper selection:", base_piper_voice)
print("Drive project:", f"{drive_root}/{voice_name}")


## 4. Resolve/download the selected base Piper voice

For Alba/Amy/other Piper keys, this reads the official `voices.json` manifest and downloads the
matching ONNX + JSON into the temporary Colab runtime. Custom Drive paths are used directly.


In [ ]:
from pathlib import Path
from urllib.parse import quote
import json, urllib.request

RUNTIME = Path("/content/rvc-piper-runtime")
BASE_DIR = RUNTIME / "base-voice"
BASE_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    "Alba Medium (UK) - en_GB-alba-medium": "en_GB-alba-medium",
    "Amy Medium (US) - en_US-amy-medium": "en_US-amy-medium",
}

def _download_file(url, destination):
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size > 0:
        return destination
    part = destination.with_suffix(destination.suffix + ".part")
    part.unlink(missing_ok=True)
    print("Downloading:", destination.name)
    urllib.request.urlretrieve(url, part)
    part.replace(destination)
    return destination

if base_piper_voice == "Custom ONNX + JSON from Drive":
    base_model = Path(custom_base_model)
    base_config = Path(custom_base_config)
    if not base_model.is_file():
        raise FileNotFoundError(f"Custom Piper ONNX not found: {base_model}")
    if not base_config.is_file():
        raise FileNotFoundError(f"Custom Piper JSON not found: {base_config}")
    selected_piper_key = "custom-drive-model"
else:
    selected_piper_key = (
        other_piper_voice_key.strip()
        if base_piper_voice == "Other Piper voice key"
        else PRESETS[base_piper_voice]
    )
    if not selected_piper_key:
        raise ValueError("Enter a Piper voice key.")

    manifest_url = "https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json"
    print("Looking up Piper voice:", selected_piper_key)
    with urllib.request.urlopen(manifest_url, timeout=60) as response:
        voices = json.load(response)

    voice = voices.get(selected_piper_key)
    if voice is None:
        matches = [key for key in voices if selected_piper_key.lower() in key.lower()][:20]
        raise KeyError(
            f"Piper voice key not found: {selected_piper_key}. "
            f"Close matches: {', '.join(matches) if matches else 'none'}"
        )

    files = list(voice.get("files", {}).keys())
    onnx_rel = next((p for p in files if p.endswith(".onnx")), None)
    json_rel = next((p for p in files if p.endswith(".onnx.json")), None)
    if not onnx_rel or not json_rel:
        raise RuntimeError(f"Voice manifest is missing ONNX/JSON files for {selected_piper_key}")

    base_model = BASE_DIR / Path(onnx_rel).name
    base_config = BASE_DIR / Path(json_rel).name
    hf_root = "https://huggingface.co/rhasspy/piper-voices/resolve/main/"
    _download_file(hf_root + quote(onnx_rel, safe="/._-") + "?download=true", base_model)
    _download_file(hf_root + quote(json_rel, safe="/._-") + "?download=true", base_config)

print("")
print("Selected base Piper voice:", selected_piper_key)
print("Base model:", base_model)
print("Base config:", base_config)


## 5. Build everything

This performs dataset generation → RVC pitch conversion → high-pitch cleanup → Piper GPU training →
best-checkpoint ONNX export.

If Colab disconnects after `last.ckpt` has been saved, reconnect, rerun setup/configuration, leave
`resume_if_possible = True`, and run this cell again.


In [ ]:
import subprocess

RUNTIME = "/content/rvc-piper-runtime"
cmd = [
    "python3",
    "/content/RVC-to-Piper-Training-APP/colab/colab_pipeline.py",
    "--repo-root", "/content/RVC-to-Piper-Training-APP",
    "--rvc-root", f"{RUNTIME}/rvc",
    "--rvc-python", f"{RUNTIME}/rvc-venv/bin/python",
    "--piper-python", f"{RUNTIME}/piper-venv/bin/python",
    "--drive-root", drive_root,
    "--voice-name", voice_name,
    "--rvc-model", rvc_model,
    "--prompts", prompt_file,
    "--prompt-limit", str(prompt_limit),
    "--pitch", str(pitch),
    "--index-rate", str(index_rate),
    "--protect", str(protect),
    "--f0-method", f0_method,
    "--batch-size", str(batch_size),
    "--max-epochs", str(max_epochs),
    "--checkpoint-every", str(checkpoint_every),
    "--num-workers", str(num_workers),
    "--base-model", str(base_model),
    "--base-config", str(base_config),
]
if rvc_index.strip():
    cmd += ["--rvc-index", rvc_index.strip()]
elif index_rate > 0:
    print("No RVC .index selected, so index_rate is being changed to 0.")
    cmd[cmd.index("--index-rate") + 1] = "0"
if not generate_dataset:
    cmd.append("--skip-dataset")
if not resume_if_possible:
    cmd.append("--no-resume")

print("Starting full Colab build...")
print("Base Piper:", selected_piper_key)
subprocess.run(cmd, check=True)


## 6. Check the exported files


In [ ]:
from pathlib import Path

project = Path(drive_root) / voice_name / "piper"
onnx_path = project / f"{voice_name}.onnx"
config_path = project / f"{voice_name}.onnx.json"

print("ONNX:", onnx_path, f"{onnx_path.stat().st_size / 1024**2:.1f} MB" if onnx_path.exists() else "MISSING")
print("JSON:", config_path, config_path.exists())

if not onnx_path.exists() or not config_path.exists():
    raise RuntimeError("The final Piper ONNX/JSON pair was not found yet.")


## 7. Test the finished standalone Piper voice

This is pure Piper inference — no RVC conversion after synthesis.


In [ ]:
#@title Test text
test_text = "Hello! This is my new standalone Piper voice running from Google Colab." #@param {type:"string"}

import subprocess
from IPython.display import Audio, display
from pathlib import Path

test_wav = Path("/content/test-custom-piper.wav")
piper_python = "/content/rvc-piper-runtime/piper-venv/bin/python"

subprocess.run([
    piper_python,
    "-m", "piper",
    "--model", str(onnx_path),
    "--config", str(config_path),
    "--output-file", str(test_wav),
    "--",
    test_text,
], check=True)

display(Audio(str(test_wav)))
print("Test WAV:", test_wav)


## Notes

- **Alba Medium is the default base voice.**
- Amy Medium is available directly from the dropdown.
- `Other Piper voice key` can use any entry from the official Piper `voices.json` collection.
- `Custom ONNX + JSON from Drive` lets you use your own compatible Piper base model.
- The base voice affects the synthetic source speech used before RVC; the final target voice still comes from your RVC model.
- Dataset/checkpoints/final ONNX stay in Google Drive; runtime dependencies and downloaded base voices live on Colab's temporary disk.
- Reduce `batch_size` if the assigned Colab GPU runs out of memory.
